# 01 - Data audit

Before training anything: **is the dataset good enough to trust a score measured on it?**

This notebook checks three things and saves the answers to `report/01_data_audit/`:

1. **Class balance** - how many `unoccupied_dirty` tables are there really?
2. **Leakage** - do frames from the same video sit in both train and validation/test?
3. **Label quality** - a contact sheet of every dirty-table label, to check by eye.

Then it builds a **group-aware split** (each video goes to exactly one split) for notebook 02.

No GPU needed.

In [ ]:
# This project's repo
REPO_URL = "https://github.com/kvarun10/Smart-Restaurant-Monitoring-System.git"

import os, sys
REPO = "/content/tablewatch"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!pip -q install roboflow 
print("code version:", os.popen(f"git -C {REPO} log -1 --format=%h_%s").read().strip())

In [ ]:
# The API key comes from Colab Secrets (key icon in the left sidebar), never from the notebook itself.
# Add a secret named ROBOFLOW_API_KEY and switch on "Notebook access".
from google.colab import userdata
from roboflow import Roboflow
import yaml, json, pathlib

WORKSPACE, PROJECT, VERSION = "varun-qlsfy", "table-detection-gomjy", 10
RAW = "/content/datasets/raw"
if not os.path.isdir(f"{RAW}/train"):
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location=RAW)

from tablewatch.tracker import normalize_state
NAMES = [normalize_state(n) for n in yaml.safe_load(open(f"{RAW}/data.yaml"))["names"]]
print(NAMES)

## 1. Run the audit on the dataset exactly as exported from Roboflow

In [ ]:
from tablewatch.data_audit import audit
REPORT_DIR = "/content/report/01_data_audit"
os.makedirs(REPORT_DIR, exist_ok=True)

report = audit(RAW, NAMES)
json.dump(report, open(f"{REPORT_DIR}/audit_original_split.json", "w"), indent=2)

import pandas as pd
rows = [{"split": s, "images": v["n_images"], **{c: d["instances"] for c, d in v["classes"].items()}}
        for s, v in report["class_table"].items()]
class_df = pd.DataFrame(rows).set_index("split")
class_df

In [ ]:
import matplotlib.pyplot as plt
ax = class_df[NAMES].T.plot.bar(figsize=(8, 4), logy=True, rot=0)
ax.set_ylabel("labelled tables (log scale)"); ax.set_title("Class balance, original split")
for c in ax.containers: ax.bar_label(c, fontsize=8)
plt.tight_layout(); plt.savefig(f"{REPORT_DIR}/class_balance_original.png", dpi=150); plt.show()

## 2. Where do the images come from?

`source_group()` guesses the source video/camera from the file name. **Read this table**: if rows
look wrong (e.g. two different cameras merged, or every image its own group), tell Claude the
file-name pattern and the grouping rule gets fixed before anything is trained.

In [ ]:
groups_df = pd.DataFrame(report["groups"]).fillna(0).set_index("group").astype(int)
groups_df.to_csv(f"{REPORT_DIR}/source_groups.csv")
print(len(groups_df), "source groups;", "largest holds", f'{groups_df.total.max() / groups_df.total.sum():.0%}', "of all images")
groups_df.head(40)

In [ ]:
print("Images whose SOURCE also appears in train:")
for split, v in report["group_leakage"].items():
    print(f'  {split}: {v["n_from_groups_also_in_train"]}/{v["n_images"]} = {v["share"]:.0%}')
print("Images with a NEAR-IDENTICAL twin in train (pixel hash, ignores file names):")
for split, v in report["near_duplicates"].items():
    print(f'  {split}: {v["n_with_near_duplicate_in_train"]}/{v["n_images"]} = {v["share"]:.0%}')

## 3. Look at every dirty-table label

`unoccupied_dirty` is the class the whole product depends on, and it is the rarest. The sheets
below show **every** labelled dirty table. Scroll through and note wrong ones (clean table
labelled dirty, box on the wrong object, the same table repeated dozens of times).

In [ ]:
from PIL import Image, ImageDraw
from tablewatch.data_audit import load_yolo_dataset
records = load_yolo_dataset(RAW)
DIRTY = NAMES.index("unoccupied_dirty")
dirty = [r for r in records if any(b[0] == DIRTY for b in r.boxes)]
print(len(dirty), "images contain a dirty-table label, from", len({r.group for r in dirty}), "source groups")

def tile(r, size=320):
    im = Image.open(r.image_path).convert("RGB"); W, H = im.size; d = ImageDraw.Draw(im)
    for cls, cx, cy, w, h in r.boxes:
        if cls == DIRTY:
            d.rectangle([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], outline="red", width=4)
    im = im.resize((size, size)); ImageDraw.Draw(im).text((4, 4), f"{r.split} | {r.group}"[:48], fill="yellow")
    return im

PER_SHEET, COLS = 30, 6
for k in range(0, len(dirty), PER_SHEET):
    chunk = dirty[k:k + PER_SHEET]
    sheet = Image.new("RGB", (COLS * 320, -(-len(chunk) // COLS) * 320))
    for i, r in enumerate(chunk): sheet.paste(tile(r), ((i % COLS) * 320, (i // COLS) * 320))
    path = f"{REPORT_DIR}/dirty_labels_{k // PER_SHEET:02d}.jpg"; sheet.save(path, quality=80)
    if k < 3 * PER_SHEET: display(sheet)

## 4. Build the group-aware split and audit it again

In [ ]:
from tablewatch.data_audit import load_yolo_dataset, class_table, leakage_report, near_duplicate_report
from tablewatch.split import group_split, write_split

SEED, RATIOS = 0, (0.7, 0.15, 0.15)
GROUPED = "/content/datasets/grouped"
records = load_yolo_dataset(RAW)
assignment = group_split(records, n_classes=len(NAMES), ratios=RATIOS, seed=SEED)
GROUPED_YAML = write_split(records, assignment, GROUPED, NAMES)
grouped_records = load_yolo_dataset(GROUPED)

new_report = audit(GROUPED, NAMES)
json.dump(new_report, open(f"{REPORT_DIR}/audit_grouped_split.json", "w"), indent=2)
json.dump({"seed": SEED, "ratios": RATIOS, "assignment": assignment}, open(f"{REPORT_DIR}/grouped_split_assignment.json", "w"), indent=2)

rows = [{"split": s, "images": v["n_images"], **{c: d["instances"] for c, d in v["classes"].items()}}
        for s, v in new_report["class_table"].items()]
display(pd.DataFrame(rows).set_index("split"))
print("leakage after regrouping:", {s: v["share"] for s, v in new_report["group_leakage"].items()},
      "| near-duplicates:", {s: v["share"] for s, v in new_report["near_duplicates"].items()})

**How to read the result**

* If valid/test now have only a handful of dirty tables (or zero), that is the honest picture:
  the dataset cannot yet prove the model detects dirty tables in a restaurant it has not seen.
  The fix is more dirty-table data from more places, not more epochs.
* If near-duplicates are still above ~5% after regrouping, the file names hide the true source;
  share `source_groups.csv` and the grouping rule gets tightened.

In [ ]:
# Download the evidence, then on your PC unzip it into the repo's report/ folder and commit it.
import shutil
from google.colab import files
shutil.make_archive("/content/01_data_audit", "zip", REPORT_DIR)
files.download("/content/01_data_audit.zip")